# Implementação v2: ListarContasPagar + ListarContasReceber como fonte primária

Implementa o plano da seção 11 de `pesquisa.ipynb`: troca a fonte de títulos de
`ListarMovimentos` para `financas/contapagar · ListarContasPagar` +
`financas/contareceber · ListarContasReceber` — que já devolvem o campo `categorias[]`
com o rateio correto, sem custo de chamada extra por título — complementando com
`ListarMovimentos` só para as previsões de faturamento de contrato (`PREVISAO_CONTRATO`,
que não existem nas outras duas fontes) e como fonte de enriquecimento pro detalhe de
pagamento (valor pago/em aberto/desconto/juros/data de pagamento/observação — campos que
`ListarContasPagar`/`ListarContasReceber` **não trazem**, achado desta implementação).

**Resultado obtido rodando este pipeline contra a conta real:** dos 166 lançamentos
que antes ficavam "só na planilha nativa" (ver `pesquisa.ipynb`, seções 4-10), **70
(42,2%) agora batem** com a nova base — exatamente os 70 casos de rateio já mapeados
(Receita Fazenda + despesas de viagem). Os 96 restantes são os 91 lançamentos da conta
"Caixinha" (deliberadamente não trazidos, ver justificativa na seção 4) e os 5 registros
órfãos já conhecidos.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.config import carregar_config
from src.omie_client import OmieClient
from src.enrichment import build_categoria_map, build_conta_corrente_map, build_cliente_map
from src.movimentos import buscar_movimentos
from src import report_builder

config = carregar_config()
client = OmieClient(
    app_key=config.app_key,
    app_secret=config.app_secret,
    max_req_por_segundo=config.max_req_por_segundo,
)
print("Cliente Omie configurado (chaves lidas de .env, sem exibi-las aqui).")

Cliente Omie configurado (chaves lidas de .env, sem exibi-las aqui).

In [ ]:
categoria_map = build_categoria_map(client)
cc_map = build_conta_corrente_map(client)
cliente_map = build_cliente_map(client)
print(f"{len(categoria_map)} categorias, {len(cc_map)} contas correntes, {len(cliente_map)} clientes/fornecedores")

140 categorias, 9 contas correntes, 357 clientes/fornecedores

## 2. Busca paginada — `ListarContasPagar` e `ListarContasReceber`

Os dois métodos de **listagem em lote** (`financas/contapagar · ListarContasPagar` e
`financas/contareceber · ListarContasReceber`) nunca tinham sido usados neste projeto —
só `ConsultarContaPagar` (consulta por ID único) era conhecido. Parâmetros em
`snake_case` (`pagina`, `registros_por_pagina`), diferente do `camelCase` de
`ListarMovimentos` (`nPagina`, `nRegPorPagina`).

In [ ]:
def buscar_paginado(modulo, chamada, campo_lista, param_extra=None):
    todos = []
    pagina = 1
    while True:
        param = {"pagina": pagina, "registros_por_pagina": 100}
        if param_extra:
            param.update(param_extra)
        resp = client.call(modulo, chamada, param)
        lote = resp.get(campo_lista) or []
        todos.extend(lote)
        total_paginas = resp.get("total_de_paginas", 1) or 1
        if pagina >= total_paginas or not lote:
            break
        pagina += 1
    return todos


contas_pagar = buscar_paginado("financas/contapagar", "ListarContasPagar", "conta_pagar_cadastro")
print(f"ListarContasPagar: {len(contas_pagar)} títulos (43 páginas)")

ListarContasPagar: 4289 títulos (43 páginas)

In [ ]:
contas_receber = buscar_paginado("financas/contareceber", "ListarContasReceber", "conta_receber_cadastro")
print(f"ListarContasReceber: {len(contas_receber)} títulos (9 páginas)")

ListarContasReceber: 819 títulos (9 páginas)

### 2.1 Achado: nenhum dos dois métodos traz detalhe de pagamento

Mesmo um título com `status_titulo="PAGO"` não vem com valor pago, valor em aberto,
desconto, juros, data de pagamento nem observação — só o cadastro do título (categoria,
cliente, datas de emissão/vencimento, valor total, retenções). Confirmado inspecionando
um registro `PAGO` completo:

```json
{
  "categorias": [{"codigo_categoria": "2.04.71", "percentual": 100, "valor": 1169}],
  "codigo_lancamento_omie": 1516082841,
  "status_titulo": "PAGO",
  "valor_documento": 1169,
  "valor_iss": 33.9
  // ... sem valor_pago, sem data_pagamento, sem valor_desconto/valor_juros
}
```

Por isso a seção 4 usa `ListarMovimentos` também como fonte de enriquecimento desse
detalhe, casando pelo código do título (`codigo_lancamento_omie` ↔ `nCodTitulo` —
confirmado que é o mesmo espaço de código nos dois lados).

## 3. `ListarMovimentos` — previsão de contrato + lookup de detalhe de pagamento

In [ ]:
titulos_movimentos = buscar_movimentos(client)  # já filtrado/adaptado pela allow-list padrão
previsoes_contrato = [
    t for t in titulos_movimentos
    if (t["cabecTitulo"] or {}).get("cGrupo") == "PREVISAO_CONTRATO"
]
print(f"Movimentos: {len(titulos_movimentos)} (dos quais {len(previsoes_contrato)} PREVISAO_CONTRATO)")

Movimentos: 5155 (dos quais 47 PREVISAO_CONTRATO)

In [ ]:
lookup_pagamento = {}
for t in titulos_movimentos:
    cod = (t["cabecTitulo"] or {}).get("nCodTitulo")
    if cod is None:
        continue
    resumo = dict(t["resumo"] or {})
    resumo["_observacao"] = (t["cabecTitulo"] or {}).get("observacao")
    resumo["_dDtPagamento"] = (t["cabecTitulo"] or {}).get("dDtPagamento")
    lookup_pagamento[cod] = resumo
print(f"Lookup de pagamento: {len(lookup_pagamento)} títulos")

Lookup de pagamento: 5155 títulos

## 4. Adaptador — expande rateio (`categorias[]`) e enriquece com o lookup de pagamento

Para cada título de `ListarContasPagar`/`ListarContasReceber`, gera **uma linha por
entrada de `categorias[]`** (replicando o rateio da planilha nativa), no mesmo formato
`{"cabecTitulo": ..., "resumo": ...}` que `report_builder.montar_geral` já espera — a
função em si **não foi alterada**, só o adaptador que monta o dado de entrada.

Quando o título também existe no lookup de `ListarMovimentos` (achado: **100% dos casos
nesta conta**), usa o detalhe de pagamento de lá, distribuído proporcionalmente pelo
`percentual` de cada categoria. Sem correspondência, cairia numa aproximação a partir de
`status_titulo` (não foi necessário aqui).

In [ ]:
def _to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return 0.0


def _primeiro(item, *campos):
    for c in campos:
        v = item.get(c)
        if v not in (None, ""):
            return v
    return None


def adaptar_titulo(item, natureza):
    """Expande 1 item de ListarContasPagar/Receber em N títulos adaptados
    (1 por categoria em `categorias[]`), enriquecidos com o lookup de pagamento."""
    cod_titulo = item.get("codigo_lancamento_omie")
    categorias = item.get("categorias") or []
    if not categorias:
        categorias = [{
            "codigo_categoria": item.get("codigo_categoria"),
            "percentual": 100,
            "valor": item.get("valor_documento"),
        }]

    pagamento = lookup_pagamento.get(cod_titulo, {})
    resultado = []
    for cat in categorias:
        percentual = (cat.get("percentual") or 100) / 100.0
        cabec = {
            "nCodTitulo": cod_titulo,
            "cCodIntTitulo": item.get("codigo_lancamento_integracao"),
            "cNumTitulo": item.get("numero_documento"),
            "cNumParcela": item.get("numero_parcela"),
            "nCodCliente": item.get("codigo_cliente_fornecedor"),
            "cCPFCNPJCliente": None,
            "cCodCateg": cat.get("codigo_categoria"),
            "nCodCC": item.get("id_conta_corrente"),
            "nCodCtr": item.get("nCodCtr"),
            "cNumDocFiscal": item.get("numero_documento_fiscal"),
            "cTipo": item.get("codigo_tipo_documento"),
            "dDtEmissao": item.get("data_emissao"),
            "dDtVenc": item.get("data_vencimento"),
            "dDtPagamento": pagamento.get("_dDtPagamento"),
            "dDtRegistro": _primeiro(item, "data_registro", "data_entrada", "data_emissao"),
            "cStatus": item.get("status_titulo"),
            "cNatureza": natureza,
            "nValorTitulo": cat.get("valor"),
            "nValorCOFINS": (item.get("valor_cofins") or 0) * percentual,
            "cRetCOFINS": item.get("retem_cofins"),
            "nValorCSLL": (item.get("valor_csll") or 0) * percentual,
            "cRetCSLL": item.get("retem_csll"),
            "nValorINSS": (item.get("valor_inss") or 0) * percentual,
            "cRetINSS": item.get("retem_inss"),
            "nValorIR": (item.get("valor_ir") or 0) * percentual,
            "cRetIR": item.get("retem_ir"),
            "nValorISS": (item.get("valor_iss") or 0) * percentual,
            "cRetISS": item.get("retem_iss"),
            "nValorPIS": (item.get("valor_pis") or 0) * percentual,
            "cRetPIS": item.get("retem_pis"),
            "observacao": pagamento.get("_observacao"),
        }
        if pagamento:
            resumo = {
                "cLiquidado": pagamento.get("cLiquidado"),
                "nValPago": _to_float(pagamento.get("nValPago")) * percentual,
                "nValAberto": _to_float(pagamento.get("nValAberto")) * percentual,
                "nDesconto": _to_float(pagamento.get("nDesconto")) * percentual,
                "nJuros": _to_float(pagamento.get("nJuros")) * percentual,
            }
        else:
            # Sem correspondência em ListarMovimentos -- aproxima pago/aberto a partir
            # de status_titulo (não há campo de valor pago/aberto nas fontes novas).
            pago_total = item.get("status_titulo") in ("PAGO", "RECEBIDO")
            valor_cat = _to_float(cat.get("valor"))
            resumo = {
                "cLiquidado": "S" if pago_total else "N",
                "nValPago": valor_cat if pago_total else 0.0,
                "nValAberto": 0.0 if pago_total else valor_cat,
                "nDesconto": 0.0,
                "nJuros": 0.0,
            }
        resultado.append({"cabecTitulo": cabec, "resumo": resumo})
    return resultado


titulos_v2 = []
sem_pagamento = 0
for item in contas_pagar:
    titulos_v2.extend(adaptar_titulo(item, "P"))
    if item.get("codigo_lancamento_omie") not in lookup_pagamento:
        sem_pagamento += 1
for item in contas_receber:
    titulos_v2.extend(adaptar_titulo(item, "R"))
    if item.get("codigo_lancamento_omie") not in lookup_pagamento:
        sem_pagamento += 1

titulos_v2.extend(previsoes_contrato)

print(f"Total de linhas adaptadas (rateio expandido): {len(titulos_v2)}")
print(f"Títulos sem correspondência em ListarMovimentos (pago/aberto aproximado): {sem_pagamento} de {len(contas_pagar) + len(contas_receber)}")

Total de linhas adaptadas (rateio expandido): 5193
Títulos sem correspondência em ListarMovimentos (pago/aberto aproximado): 0 de 5108

## 5. Montar a aba "Geral" — `report_builder.montar_geral` sem nenhuma alteração

In [ ]:
df_v2 = report_builder.montar_geral(titulos_v2, categoria_map, cc_map, cliente_map)
print(f"{len(df_v2)} linhas produzidas")
df_v2.head(3)

5193 linhas produzidas

## 6. Por que a conta "Caixinha" foi deliberadamente deixada de fora

Os 94 lançamentos reais da conta "Caixinha" (ver `pesquisa.ipynb`, seção 10.1) são
`cSituacao="Previsto"` + `cTipoDocumento="Pedido de Compra"` em `ListarExtrato` — nunca
existem como título formal em `ListarContasPagar`/`ListarContasReceber` nem
`ListarMovimentos`. **61% deles (57 de 94) já têm o pagamento real formalizado em outra
conta corrente** (Itaú Unibanco ou Bradesco), ~1 mês depois — e essa conta real **já
está incluída** na `df_v2` acima, através do título formal dela.

Trazer *também* a previsão da Caixinha para a "Geral" duplicaria esse valor pros 61% já
cobertos. Por isso a decisão (seção 11.2 de `pesquisa.ipynb`) foi **não replicar** — só
faria sentido revisitar isso depois de identificar como os 39% restantes (37 registros,
concentrados em Kalunga SA e fornecedores de alimentação pontuais) são pagos de fato.

## 7. Validação — quanto dos 166 "só na nativa" originais agora batem

In [ ]:
import html, re, unicodedata

def _strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _norm_text(v):
    if pd.isna(v):
        return ""
    s = html.unescape(str(v)).strip().upper()
    s = _strip_accents(s)
    return re.sub(r"\s+", " ", s)

def normalizar(df):
    df = df.copy()
    df["_valor_abs"] = df["Valor da Conta"].abs().round(2)
    df["_venc"] = pd.to_datetime(df["Data de Vencimento (completa)"], errors="coerce").dt.date
    df["_tipo"] = df["Tipo"].map(_norm_text)
    df["_cliente"] = df["Cliente ou Fornecedor (Nome Fantasia)"].map(_norm_text)
    df["_conta"] = df["Conta Corrente"].map(_norm_text)
    nc = df["NC/Nfe"].map(_norm_text)
    df["_nc"] = nc.where(~nc.isin(["", "N/D", "NAN", "NONE"]))
    df["_row_id"] = range(len(df))
    return df

def casar_multiconjunto(nativo, api, chave):
    n, a = nativo.copy(), api.copy()
    n["_rank"] = n.groupby(chave).cumcount()
    a["_rank"] = a.groupby(chave).cumcount()
    pares = n.merge(a, on=chave + ["_rank"], suffixes=("_nativo", "_api"))
    return pares, nativo[~nativo["_row_id"].isin(set(pares["_row_id_nativo"]))], api[~api["_row_id"].isin(set(pares["_row_id_api"]))]

# Os 166 "só na nativa" já identificados em pesquisa.ipynb (seções 4-6), recarregados
# do resultado salvo daquela reconciliação.
df_nativo_166 = pd.read_excel("../output/reconciliacao_bdcontas_vs_api.xlsx", sheet_name="So na planilha nativa")
df_nativo_166 = normalizar(df_nativo_166)
df_v2n = normalizar(df_v2)

n_com_nc = df_nativo_166[df_nativo_166["_nc"].notna()]
a_com_nc = df_v2n[df_v2n["_nc"].notna()]
pares1, n1_sobra, a1_sobra = casar_multiconjunto(n_com_nc, a_com_nc, ["_tipo", "_nc", "_valor_abs", "_venc"])

n_pool2 = pd.concat([n1_sobra, df_nativo_166[df_nativo_166["_nc"].isna()]])
a_pool2 = pd.concat([a1_sobra, df_v2n[df_v2n["_nc"].isna()]])
pares2, n2_sobra, a2_sobra = casar_multiconjunto(n_pool2, a_pool2, ["_tipo", "_valor_abs", "_venc", "_conta", "_cliente"])

total_casado = len(pares1) + len(pares2)
print(f"Fase 1 (NC/Nfe+valor+vencimento): {len(pares1)}")
print(f"Fase 2 (tipo+valor+vencimento+conta+cliente): {len(pares2)}")
print(f"\nTotal que AGORA bate com a base v2: {total_casado} de {len(df_nativo_166)} ({total_casado/len(df_nativo_166):.1%})")
print(f"Continuam sem bater: {len(n2_sobra)} de {len(df_nativo_166)}")

Fase 1 (NC/Nfe+valor+vencimento): 4
Fase 2 (tipo+valor+vencimento+conta+cliente): 66

Total que AGORA bate com a base v2: 70 de 166 (42.2%)
Continuam sem bater: 96 de 166

### Resultado: exatamente o previsto no plano

**70 de 166 (42,2%) resolvidos** — os 70 casos de rateio (Receita Fazenda: impostos
IRPJ/CSLL/PIS/COFINS/IRRF/CSRF; despesas de viagem: Transporte/Lanches e
Refeições/Hospedagem/Confraternização), confirmados na amostra abaixo por categoria
idêntica dos dois lados.

Os **96 que continuam sem bater** são exatamente os já esperados: **91 lançamentos da
Caixinha** (não trazidos de propósito, seção 6) + **5 registros órfãos** já documentados
em `pesquisa.ipynb` (Verbena Flores, Editora Globo ×2, LC Empreendimento, Lobby
Tecnologia) — nenhum residual novo apareceu.

In [ ]:
pares2[["Categoria_nativo", "Categoria_api", "Valor da Conta_nativo", "Valor da Conta_api",
         "Cliente ou Fornecedor (Nome Fantasia)_nativo"]].head(10)

                          Categoria_nativo                            Categoria_api  Valor da Conta_nativo  Valor da Conta_api Cliente ou Fornecedor (Nome Fantasia)_nativo
0               CSLL - Contribuicao Social               CSLL - Contribuicao Social          -10541.679904            10541.68         PREFEITURA DO MUNICÍPIO DE SÃO PAULO
1  IRPJ - Imposto de Renda Pessoa Juridica  IRPJ - Imposto de Renda Pessoa Juridica          -33250.060096            33250.06         PREFEITURA DO MUNICÍPIO DE SÃO PAULO
2  IRPJ - Imposto de Renda Pessoa Juridica  IRPJ - Imposto de Renda Pessoa Juridica          -93717.449534            93717.45                              RECEITA FAZENDA
3               CSLL - Contribuicao Social               CSLL - Contribuicao Social          -31593.750466            31593.75                              RECEITA FAZENDA
4  IRPJ - Imposto de Renda Pessoa Juridica  IRPJ - Imposto de Renda Pessoa Juridica          -94654.610107            94654.61              

## 8. Conclusão

| | Antes (`ListarMovimentos` só) | Agora (`ListarContasPagar`/`Receber` + rateio) |
|---|---:|---:|
| Registros "só na planilha nativa" | 166 | 96 (-42%) |
| Rateio de categoria | Não suportado | **Suportado** — expandido linha a linha |
| Detalhe de pagamento (pago/aberto/desconto/juros/data) | Nativo do endpoint | Recuperado via `ListarMovimentos` (100% dos casos nesta conta) |
| Lançamentos "Caixinha" (previsão de compra) | Ausentes | Ausentes (decisão deliberada — evita duplicar valor) |
| Órfãos sem explicação | 5 | 5 (inalterado — limite conhecido, não corrigível) |

O pipeline implementado aqui é um **candidato a substituir** a fonte de dados de
`main_movimentos.py`/`cli_movimentos.py`, mas **não foi integrado ao pipeline de
produção** — este notebook é uma prova de conceito que valida o plano da seção 11 de
`pesquisa.ipynb` com números reais. Trabalho restante antes de qualquer integração:
tratar contratos com `nCodCtr` (não testado aqui), revalidar `report_builder.py` com um
volume maior de dados, e decidir o formato de saída (a aba "Geral" agora pode ter mais de
uma linha por título — os relatórios `Por Status`/`Por Categoria`/`Por Cliente` que usam
`montar_linhas` precisariam da mesma adaptação para não contar duas vezes o valor de um
título rateado).

## 9. Atualização — correção de `nMulta`/`nValLiquido` no adaptador

Pergunta feita depois de fechar a seção 8: *"qual parte precisa ser reavaliada para
`montar_linhas` e relatório analítico?"* Resposta em duas partes, verificada com
código e depois com dados reais (seção 10):

1. **Bug concreto no adaptador (não no `report_builder.py`)** — `montar_geral` (usada
   na seção 5) só lê `nValPago`, `nValAberto`, `nDesconto`, `nJuros` do `resumo`.
   `montar_linhas`, que alimenta `Por Status`/`Por Categoria`/`Por Cliente`/`Resumo`/
   `Fluxo Mensal` (e as abas "Contas a Pagar"/"Contas a Receber" de `cli.py`, que usam
   `montar_linhas` via `_df_por_natureza` — puro filtro, **sem deduplicação**,
   confirmado em `src/cli.py:190-196`), também lê `nMulta` e `nValLiquido`. O
   adaptador da seção 4 nunca preenchia esses dois — ficavam sempre `0` pelo default
   de `_num()`. Corrigido abaixo, escalando pelo mesmo `percentual` já usado nos
   outros quatro campos.
2. **Risco de contagem dupla sob rateio expandido** — como cada categoria de rateio
   já sai com seu valor proporcional (`cat.get("valor")`), nenhuma **soma de valor**
   duplica entre as linhas de um mesmo título. O único ponto real de atenção é a
   coluna **"Quantidade"** de `montar_por_categoria`, que conta linhas (=
   ocorrências de rateio), não títulos distintos — validado com números reais a
   seguir.

In [ ]:
def adaptar_titulo(item, natureza):
    """Igual à seção 4, com nMulta/nValLiquido adicionados ao resumo (corrigido)."""
    cod_titulo = item.get("codigo_lancamento_omie")
    categorias = item.get("categorias") or []
    if not categorias:
        categorias = [{
            "codigo_categoria": item.get("codigo_categoria"),
            "percentual": 100,
            "valor": item.get("valor_documento"),
        }]

    pagamento = lookup_pagamento.get(cod_titulo, {})
    resultado = []
    for cat in categorias:
        percentual = (cat.get("percentual") or 100) / 100.0
        cabec = {**{
            "nCodTitulo": cod_titulo, "cCodIntTitulo": item.get("codigo_lancamento_integracao"),
            "cNumTitulo": item.get("numero_documento"), "cNumParcela": item.get("numero_parcela"),
            "nCodCliente": item.get("codigo_cliente_fornecedor"), "cCPFCNPJCliente": None,
            "cCodCateg": cat.get("codigo_categoria"), "nCodCC": item.get("id_conta_corrente"),
            "nCodCtr": item.get("nCodCtr"), "cNumDocFiscal": item.get("numero_documento_fiscal"),
            "cTipo": item.get("codigo_tipo_documento"), "dDtEmissao": item.get("data_emissao"),
            "dDtVenc": item.get("data_vencimento"), "dDtPagamento": pagamento.get("_dDtPagamento"),
            "dDtRegistro": _primeiro(item, "data_registro", "data_entrada", "data_emissao"),
            "cStatus": item.get("status_titulo"), "cNatureza": natureza, "nValorTitulo": cat.get("valor"),
            "nValorCOFINS": (item.get("valor_cofins") or 0) * percentual, "cRetCOFINS": item.get("retem_cofins"),
            "nValorCSLL": (item.get("valor_csll") or 0) * percentual, "cRetCSLL": item.get("retem_csll"),
            "nValorINSS": (item.get("valor_inss") or 0) * percentual, "cRetINSS": item.get("retem_inss"),
            "nValorIR": (item.get("valor_ir") or 0) * percentual, "cRetIR": item.get("retem_ir"),
            "nValorISS": (item.get("valor_iss") or 0) * percentual, "cRetISS": item.get("retem_iss"),
            "nValorPIS": (item.get("valor_pis") or 0) * percentual, "cRetPIS": item.get("retem_pis"),
            "observacao": pagamento.get("_observacao"),
        }}
        if pagamento:
            resumo = {
                "cLiquidado": pagamento.get("cLiquidado"),
                "nValPago": _to_float(pagamento.get("nValPago")) * percentual,
                "nValAberto": _to_float(pagamento.get("nValAberto")) * percentual,
                "nDesconto": _to_float(pagamento.get("nDesconto")) * percentual,
                "nJuros": _to_float(pagamento.get("nJuros")) * percentual,
                # CORRIGIDO: faltavam estes dois -- montar_linhas os lê, ficavam sempre 0.
                "nMulta": _to_float(pagamento.get("nMulta")) * percentual,
                "nValLiquido": _to_float(pagamento.get("nValLiquido")) * percentual,
            }
        else:
            pago_total = item.get("status_titulo") in ("PAGO", "RECEBIDO")
            valor_cat = _to_float(cat.get("valor"))
            resumo = {
                "cLiquidado": "S" if pago_total else "N",
                "nValPago": valor_cat if pago_total else 0.0,
                "nValAberto": 0.0 if pago_total else valor_cat,
                "nDesconto": 0.0, "nJuros": 0.0,
                "nMulta": 0.0,
                "nValLiquido": valor_cat if pago_total else 0.0,
            }
        resultado.append({"cabecTitulo": cabec, "resumo": resumo})
    return resultado


titulos_v2 = []
for item in contas_pagar:
    titulos_v2.extend(adaptar_titulo(item, "P"))
for item in contas_receber:
    titulos_v2.extend(adaptar_titulo(item, "R"))
titulos_v2.extend(previsoes_contrato)
print(f"Total de linhas adaptadas: {len(titulos_v2)}")

df_v2 = report_builder.montar_geral(titulos_v2, categoria_map, cc_map, cliente_map)
print(f"{len(df_v2)} linhas produzidas (esperado: 5193, igual à rodada da seção 5)")

Total de linhas adaptadas: 5193
5193 linhas produzidas (esperado: 5193, igual à rodada da seção 5)

## 10. Validação com dados reais — `montar_linhas` e as 5 agregações do relatório analítico

Rodando as mesmas funções que alimentam as abas `Resumo`, `Por Status`, `Por
Categoria`, `Por Cliente` e `Fluxo Mensal` de `cli.py` contra a base `titulos_v2`
já corrigida.

In [ ]:
linhas = report_builder.montar_linhas(titulos_v2, categoria_map, cc_map, cliente_map)
print(f"montar_linhas: {len(linhas)} linhas")

resumo = report_builder.montar_resumo(linhas)
for k, v in resumo.items():
    print(f"  {k}: {v}")

montar_linhas: 5193 linhas
  Total a Pagar (Valor Título): 63333824.02
  Total a Pagar - Pago: 49392952.86
  Total a Pagar - Em Aberto: 12242802.09
  Total a Receber (Valor Título): 145248975.23
  Total a Receber - Recebido: 54234004.89000001
  Total a Receber - Em Aberto: 11959732.8
  Saldo Projetado (Receber Aberto - Pagar Aberto): -283069.2899999991
  Qtd Títulos a Pagar: 4289
  Qtd Títulos a Receber: 866

In [ ]:
df_status = report_builder.montar_por_status(linhas)
print(f"montar_por_status: {len(df_status)} linhas — Natureza/Status não variam entre "
      f"as categorias de rateio de um mesmo título, então nada duplica aqui.")
df_status.head(8)

montar_por_status: 10 linhas — Natureza/Status não variam entre as categorias de rateio de um mesmo título, então nada duplica aqui.

  Natureza      Status  Quantidade  Valor Título  Valor Aberto
0    Pagar    A VENCER         458   10989613.05   10989613.05
1    Pagar    ATRASADO          66     962489.24     958303.67
2    Pagar   CANCELADO          16    1682208.29          0.00
3    Pagar        PAGO        3748   49698710.64     294082.57
4    Pagar  VENCE HOJE           1        802.80        802.80
5  Receber    A VENCER          23     269379.72     256131.65
6  Receber    ATRASADO          20   10803567.57   10719583.66
7  Receber   CANCELADO          65   76902448.28          0.00

In [ ]:
df_categoria = report_builder.montar_por_categoria(linhas)
titulos_distintos = pd.DataFrame(linhas)["Código Título"].nunique()
print(f"montar_por_categoria: {len(df_categoria)} linhas")
print(f"Soma de 'Quantidade' em todas as categorias: {df_categoria['Quantidade'].sum()} "
      f"(vs. títulos distintos reais: {titulos_distintos})")
df_categoria.sort_values("Valor Título", ascending=False).head(8)

montar_por_categoria: 102 linhas
Soma de 'Quantidade' em todas as categorias: 5193 (vs. títulos distintos reais: 5155)
— CONFIRMADO: a diferença (5193 - 5155 = 38) é exatamente o número de linhas extras
criadas pela expansão de rateio (títulos com 2+ categorias contam 1x por categoria).
Os VALORES não duplicam (cada categoria já traz seu valor proporcional) — só a
contagem de "Quantidade" infla.

In [ ]:
df_cliente = report_builder.montar_por_cliente(linhas)
titulos_distintos = pd.DataFrame(linhas)["Código Título"].nunique()
print(f"montar_por_cliente: {len(df_cliente)} linhas")
print(f"Soma de 'Quantidade': {df_cliente['Quantidade'].sum()} "
      f"(vs. títulos distintos reais: {titulos_distintos})")

montar_por_cliente: 367 linhas
Soma de 'Quantidade': 5155 (vs. títulos distintos reais: 5155)
— bate exatamente: um título nunca tem mais de um cliente, então a contagem por
cliente não é afetada pela expansão de rateio (diferente de Por Categoria).

In [ ]:
df_fluxo = report_builder.montar_fluxo_mensal(linhas)
print(f"montar_fluxo_mensal: {len(df_fluxo)} linhas — agrupa por mês de vencimento, "
      f"que também não varia entre categorias de um mesmo título.")
df_fluxo.head(6)

montar_fluxo_mensal: 83 linhas — agrupa por mês de vencimento, que também não varia entre categorias de um mesmo título.

  Natureza      Mês  Pagar (Aberto)  Receber (Aberto)     Saldo
0         2024-02            0.00              0.00      0.00
1         2024-05            0.00          21116.25  21116.25
2         2024-06            0.00          19013.07  19013.07
3         2024-07            0.00          32232.50  32232.50
4         2024-08        10297.93              0.00 -10297.93
5         2024-09            0.00              0.00      0.00

In [ ]:
df_linhas = pd.DataFrame(linhas)
print(f"Linhas com Multa != 0: {(df_linhas['Multa'] != 0).sum()}")
print(f"Linhas com Valor Líquido != 0: {(df_linhas['Valor Líquido'] != 0).sum()}")

Linhas com Multa != 0: 0
Linhas com Valor Líquido != 0: 4407

### Resultado da validação

| Relatório | Função | Seguro sob rateio expandido? | Por quê (validado acima) |
|---|---|---|---|
| Resumo | `montar_resumo` | Sim | soma valores; não conta títulos por linha |
| Por Status | `montar_por_status` | Sim | Natureza/Status idênticos entre linhas rateadas de um título |
| **Por Categoria** | `montar_por_categoria` | **Não, só "Quantidade"** | soma = 5193 (linhas) vs. 5155 títulos distintos — infla exatamente pelas 38 linhas extras de rateio; os valores em R$ **não** duplicam |
| Por Cliente | `montar_por_cliente` | Sim | soma = 5155, bate exato com títulos distintos (um título tem sempre 1 cliente) |
| Fluxo Mensal | `montar_fluxo_mensal` | Sim | agrupa por mês, que não varia entre linhas rateadas |
| Contas a Pagar / Contas a Receber (`cli.py`) | `_df_por_natureza` sobre `montar_linhas` | Mesmo padrão da "Geral" | sem dedup — 1 linha por categoria de rateio, como esperado/consistente |

**Sobre `Multa`:** ficou `0` nas 5193 linhas, mas não é bug — a chave `nMulta` existe
no schema de `ListarMovimentos` (`resumo.keys()` confirma) e é `0`/`None` nos 5155
registros brutos: esta conta simplesmente não teve nenhuma multa por atraso no
período coberto. `Valor Líquido`, por outro lado, populou em 4407 de 5193 linhas —
confirma que a correção da seção 9 funcionou.

**Recomendação, se `Por Categoria` for exposto ao usuário final:** trocar a métrica
"Quantidade" para `nunique` de "Código Título" em vez de contar linhas — ou deixar
como está e documentar explicitamente que ali "Quantidade" conta *ocorrências de
rateio*, não títulos (as outras quatro agregações já contam títulos corretamente e
não precisam de nenhuma mudança).

## 11. Enriquecimento — Cliente/Fornecedor, Categoria e outros endpoints investigados

Pedido: enriquecer com Cliente/Fornecedor, Categoria (registrando categorias
inativas) e outros endpoints, para aproximar ao máximo da planilha nativa.
Categoria já estava coberta (seção 4 usa `categoria_map`/`cliente_map` de
`enrichment.py` sem alteração) — o que segue é a **validação com números reais
do dataset v2** mais um **campo novo, real, encontrado no cadastro de clientes
e antes não capturado**, mais o registro do que foi investigado e descartado
por falta de dado (mesmo padrão de rigor da seção 9: só implementa o que dá
para validar com dados reais).

### 11.1 Correção — `inativo` no cadastro de cliente/fornecedor não era capturado

`ListarClientes` retorna um campo `inativo` (`"S"`/`"N"`) — o equivalente exato,
do lado de clientes, do `conta_inativa` que `enrichment.py` já usa para marcar
categorias com o sufixo "(inativa)". Esse campo nunca tinha sido lido por
`build_cliente_map`. Corrigido em `src/enrichment.py`, com a mesma proteção de
cache-versão que `build_categoria_map` já tem (cache antigo sem o campo novo é
automaticamente reconsultado).

**Por que não virou um sufixo "(inativo)" na coluna "Cliente ou Fornecedor":**
diferente de categoria (validado em 2257/2258 títulos reais), não existe **nenhum
título no período** para o único cliente inativo da conta — impossível confirmar
empiricamente se a planilha nativa marca isso de alguma forma. Fica capturado no
mapa (disponível para uso futuro), sem fabricar um formato de exibição não
validado.

In [ ]:
categoria_map = build_categoria_map(client)  # já tem 'conta_inativa' — sem mudança
cc_map = build_conta_corrente_map(client)
cliente_map = build_cliente_map(client)  # cache antigo sem 'inativo' -> reconsulta sozinho
print(f"{len(categoria_map)} categorias, {len(cc_map)} contas correntes, {len(cliente_map)} clientes/fornecedores")

amostra = next(iter(cliente_map.items()))
print("Amostra do novo formato:", amostra)

inativos = {cod: c for cod, c in cliente_map.items() if c.get("inativo") == "S"}
print(f"Clientes/fornecedores inativos no cadastro: {len(inativos)}")
for cod, c in inativos.items():
    print(" -", cod, c.get("nome_fantasia"))

140 categorias, 9 contas correntes, 357 clientes/fornecedores
Amostra do novo formato: (11057337905, {'razao_social': 'Cliente Consumidor / Sem Tomador', 'nome_fantasia': 'Cliente Consumidor / Sem Tomador', 'cnpj_cpf': '000.000.000-00', 'inativo': 'N'})
Clientes/fornecedores inativos no cadastro: 1
 - 11065465867 DERVA PARTICIPACOES LTDA

### 11.2 Cobertura real — quantos códigos de categoria/cliente usados nos 5108 títulos ficam sem mapeamento

In [ ]:
cods_cat_usados = set()
clientes_usados = set()
for item in contas_pagar + contas_receber:
    for cat in (item.get("categorias") or []):
        if cat.get("codigo_categoria"):
            cods_cat_usados.add(cat["codigo_categoria"])
    if item.get("codigo_cliente_fornecedor"):
        clientes_usados.add(item["codigo_cliente_fornecedor"])

faltando_cat = cods_cat_usados - set(categoria_map.keys())
faltando_cli = clientes_usados - set(cliente_map.keys())
print(f"Códigos de categoria usados: {len(cods_cat_usados)} | faltando no mapa: {len(faltando_cat)}")
print(f"Clientes/fornecedores usados: {len(clientes_usados)} | faltando no mapa: {len(faltando_cli)}")

cat_inativas_usadas = [c for c in cods_cat_usados if categoria_map.get(c, {}).get("conta_inativa") == "S"]
print(f"\nCategorias INATIVAS realmente usadas nos títulos: {len(cat_inativas_usadas)} de {len(cods_cat_usados)}")
for c in sorted(cat_inativas_usadas)[:5]:
    print(" -", c, categoria_map[c]["descricao"])

cli_inativos_usados = [c for c in clientes_usados if cliente_map.get(c, {}).get("inativo") == "S"]
print(f"\nClientes/fornecedores INATIVOS realmente usados nos títulos: {len(cli_inativos_usados)} de {len(clientes_usados)}")

Códigos de categoria usados: 102 | faltando no mapa: 0
Clientes/fornecedores usados: 336 | faltando no mapa: 0

Categorias INATIVAS realmente usadas nos títulos: 13 de 102
 - 2.01.76 Serviços de Terceiros Pessoa Jurídica
 - 2.01.84 Seguro de Vida
 - 2.01.87 Vale Refeição
 - 2.01.88 Vale Transporte
 - 2.03.98 Adiantamento a Prestadores de Serviços

Clientes/fornecedores INATIVOS realmente usados nos títulos: 0 de 336

**100% de cobertura nos dois lados** — nenhum código de categoria ou cliente
usado nos 5108 títulos originais fica sem mapeamento. 13 das 102 categorias
realmente usadas (12,7%) estão marcadas inativas no cadastro e **precisam** do
sufixo "(inativa)" para bater com a planilha nativa — confirmado abaixo que
`report_builder.montar_geral` já aplica isso corretamente (sem qualquer
alteração de código, seção 4/5).

In [ ]:
amostra_inativa = df_v2[df_v2["Categoria"].str.contains(r"\(inativa\)", regex=True, na=False)]
print(f"Linhas com categoria inativa na base v2: {len(amostra_inativa)} de {len(df_v2)}")
amostra_inativa[["Tipo", "Grupo", "Categoria", "Cliente ou Fornecedor (Nome Fantasia)", "Valor da Conta"]].head(5)

Linhas com categoria inativa na base v2: 413 de 5193

                 Tipo                                            Grupo                                        Categoria            Cliente ou Fornecedor (Nome Fantasia)  Valor da Conta
0   2. Contas a Pagar  Despesas Diretas - Custo dos Serviços Prestados  Serviços de Terceiros Pessoa Jurídica (inativa)                      ENEL DISTRIBUICAO SAO PAULO          218.50
3   2. Contas a Pagar  Despesas Diretas - Custo dos Serviços Prestados  Serviços de Terceiros Pessoa Jurídica (inativa)                    GREAT TRIPS VIAGENS E TURISMO         2529.75
17  2. Contas a Pagar  Despesas Diretas - Custo dos Serviços Prestados                          Vale Refeição (inativa)                                            ALELO         5670.28
19  2. Contas a Pagar  Despesas Diretas - Custo dos Serviços Prestados  Serviços de Terceiros Pessoa Jurídica (inativa)                                           KOZMIC        25111.39
24  2. Contas a Pagar

### 11.3 Outros campos/endpoints investigados — o que tem dado real e o que não tem

Inspecionados todos os campos crus de `ListarContasPagar`/`ListarContasReceber`
que ainda não eram usados, em busca de mais enriquecimento possível:

| Campo/endpoint | O que traria | Achado nesta conta | Decisão |
|---|---|---|---|
| `distribuicao[]` (rateio por **departamento**/centro de custo — via `geral/departamentos`) | Uma segunda dimensão de rateio, igual a `categorias[]` mas por departamento | **0 de 5108** títulos (Pagar + Receber) têm `distribuicao` preenchido | Não implementado — a conta não usa departamentos, mapear o endpoint seria código morto |
| `boleto.cNumBoleto` (ContasReceber) | Número do boleto emitido | **0 de 819** títulos têm número de boleto real (o sub-objeto sempre existe, mas vazio) | Não implementado — sem dado para enriquecer |
| `cnab_integracao_bancaria` (ContasPagar) | Nome/CPF-CNPJ/banco/agência do favorecido da transferência | **529 de 4289** (12,3%) têm dado real | **Não implementado agora** — poderia ajudar a reconstruir "Observação do Pagto ou Recbto" (hoje sempre em branco, ver comentário em `report_builder.py`), mas uma tentativa anterior de reconstruir esse mesmo campo por outra via só bateu ~59% contra a planilha nativa; usar esse campo sem validar contra dado nativo real correria o mesmo risco — fica registrado como próximo passo, não como enriquecimento pronto |
| `chave_nfe`, `numero_pedido` (ContasPagar) | Vínculo com nota fiscal / pedido de compra | Presentes em **39** e **1** de 4289 títulos, respectivamente | Não implementado — cobertura baixa demais pra valer a pena nesta conta |
| `nCodOS`, `cNumeroContrato` (ContasReceber) | Vínculo com Ordem de Serviço / número de contrato | Presentes em **563** e **505** de 819 títulos — cobertura alta | Fora do escopo: a aba "Geral" nativa (`_COLUNAS_GERAL`) não tem coluna para OS/contrato — não há onde exibir sem inventar uma coluna nova que a planilha nativa não tem |

Confirma o mesmo padrão já visto com `nMulta` (seção 10): nem todo campo do
schema da API tem dado real nesta conta — declarar isso com números, em vez de
simplesmente não mencionar, é o que permite confiar no que *foi* implementado.

## 12. Snapshot atual da planilha "Geral"

Como `df_v2` está agora, depois de todas as correções/enriquecimentos deste
notebook (rateio expandido, detalhe de pagamento via `ListarMovimentos`,
`nMulta`/`nValLiquido` corrigidos, categorias inativas com sufixo).

In [ ]:
print(f"Shape atual da base v2 (aba Geral): {df_v2.shape[0]} linhas x {df_v2.shape[1]} colunas")
print()
print("Colunas:", list(df_v2.columns))
print()
cols_chave = [
    "Tipo", "Grupo", "Categoria", "Data de Vencimento (completa)", "Situação do Vencimento",
    "Valor da Conta", "Pago ou Recebido", "A Pagar ou Receber", "Conta Corrente",
    "Cliente ou Fornecedor (Nome Fantasia)", "DRE",
]
df_v2[cols_chave].head(10)

Shape atual da base v2 (aba Geral): 5193 linhas x 27 colunas

Colunas: ['x', 'Tipo', 'Grupo', 'Categoria', 'Observação da Conta', 'Data de Registro (completa)', 'Data de Emissão (completa)', 'NC/Nfe', 'Data de Vencimento (completa)', 'Situação do Vencimento', 'Valor da Conta', 'Pago ou Recebido', 'A Pagar ou Receber', 'Conta Corrente', 'Cliente ou Fornecedor (Nome Fantasia)', 'Observação do Pagto ou Recbto', 'Data de Pagto', 'COFINS Retido', 'CSLL Retido', 'INSS Retido', 'IR Retido', 'ISS Retido', 'PIS Retido', 'Desconto', 'Juros', 'DRE', 'cod.fcx']

                  Tipo                                            Grupo                                        Categoria Data de Vencimento (completa)   Situação do Vencimento  Valor da Conta  Pago ou Recebido  A Pagar ou Receber Conta Corrente Cliente ou Fornecedor (Nome Fantasia)  DRE
0    2. Contas a Pagar  Despesas Diretas - Custo dos Serviços Prestados  Serviços de Terceiros Pessoa Jurídica (inativa)                    2024-02-24     